In [2]:
# ============================================================
# EXPERIMENT 2 — SIAMESE U-NET + CHANGE CLASSIFICATION
# + CAPTION GENERATION
# ============================================================

import os
import json
import random
import ast
import math
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1. CONFIG
# ============================================================

CSV_PATH = r"C:\College\Dissertation\Data\FINAL_LEVIR_DATASET\final_dataset.csv"

BATCH_SIZE = 4
EPOCHS = 70

LR = 1e-4
WEIGHT_DECAY = 1e-5

SEG_WEIGHT = 2.0
CLS_WEIGHT = 1.0
CAP_WEIGHT = 1.0

MAX_CAPTION_LEN = 40

SEED = 42

BEST_MODEL_PATH = "experiment2_best_model.pth"
HISTORY_PATH = "experiment2_training_history.csv"

# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)

# Don't call torch.cuda.manual_seed_all before checking CUDA
# because your previous crashed CUDA context can make this fail.

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True

print("=" * 70)
print("DEVICE")
print("=" * 70)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 3. LOAD DATA
# ============================================================

df = pd.read_csv(CSV_PATH)

print("\nDataset:")
print("Total:", len(df))
print(df["split"].value_counts())

print("\nChange distribution:")
print(
    df.groupby(["split", "changeflag"])
      .size()
)

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)

# ============================================================
# 4. VOCABULARY
# ============================================================

PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
UNK_TOKEN = "<UNK>"

special_tokens = [
    PAD_TOKEN,
    SOS_TOKEN,
    EOS_TOKEN,
    UNK_TOKEN
]

word_freq = {}

def parse_captions(x):

    if isinstance(x, list):
        return x

    try:
        return json.loads(x)
    except:
        try:
            return ast.literal_eval(x)
        except:
            return [str(x)]


for captions in train_df["captions"]:

    captions = parse_captions(captions)

    for caption in captions:

        words = str(caption).lower().split()

        for word in words:
            word_freq[word] = word_freq.get(word, 0) + 1


itos = special_tokens + sorted(word_freq.keys())

stoi = {
    word: idx
    for idx, word in enumerate(itos)
}

PAD_IDX = stoi[PAD_TOKEN]
SOS_IDX = stoi[SOS_TOKEN]
EOS_IDX = stoi[EOS_TOKEN]
UNK_IDX = stoi[UNK_TOKEN]

VOCAB_SIZE = len(itos)

print("\nVocabulary size:", VOCAB_SIZE)

# ============================================================
# 5. TOKENIZER
# ============================================================

def encode_caption(caption):

    words = str(caption).lower().split()

    tokens = [SOS_IDX]

    for word in words:
        tokens.append(
            stoi.get(word, UNK_IDX)
        )

    tokens.append(EOS_IDX)

    tokens = tokens[:MAX_CAPTION_LEN]

    if len(tokens) < MAX_CAPTION_LEN:

        tokens += [
            PAD_IDX
        ] * (
            MAX_CAPTION_LEN - len(tokens)
        )

    return torch.tensor(
        tokens,
        dtype=torch.long
    )


# ============================================================
# 6. DATASET
# ============================================================

class LEVIRDataset(Dataset):

    def __init__(self, dataframe):

        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # ----------------------------------------------------
        # Images
        # ----------------------------------------------------

        image_a = Image.open(
            row["cc_A"]
        ).convert("RGB")

        image_b = Image.open(
            row["cc_B"]
        ).convert("RGB")

        image_a = np.array(image_a).astype(
            np.float32
        ) / 255.0

        image_b = np.array(image_b).astype(
            np.float32
        ) / 255.0

        image_a = torch.from_numpy(
            image_a
        ).permute(2, 0, 1)

        image_b = torch.from_numpy(
            image_b
        ).permute(2, 0, 1)

        # ----------------------------------------------------
        # Mask
        # ----------------------------------------------------

        mask = Image.open(
            row["mask"]
        ).convert("L")

        mask = np.array(mask)

        # If original mask is 1024x1024,
        # resize to the 256x256 CC tile size.
        if mask.shape != (256, 256):

            mask = Image.fromarray(
                mask
            ).resize(
                (256, 256),
                Image.Resampling.NEAREST
            )

            mask = np.array(mask)

        mask = (
            mask > 127
        ).astype(np.float32)

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0)

        # ----------------------------------------------------
        # Change flag
        # ----------------------------------------------------

        changeflag = torch.tensor(
            float(row["changeflag"]),
            dtype=torch.float32
        )

        # ----------------------------------------------------
        # Caption
        # ----------------------------------------------------

        captions = parse_captions(
            row["captions"]
        )

        caption = captions[0]

        caption = encode_caption(
            caption
        )

        return {
            "image_a": image_a,
            "image_b": image_b,
            "mask": mask,
            "changeflag": changeflag,
            "caption": caption
        }


# ============================================================
# 7. DATA LOADERS
# ============================================================

train_dataset = LEVIRDataset(
    train_df
)

val_dataset = LEVIRDataset(
    val_df
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("\nTrain samples:", len(train_dataset))
print("Val samples:", len(val_dataset))

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Val batches:",
    len(val_loader)
)

# ============================================================
# 8. DOUBLE CONV
# ============================================================

class DoubleConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            )
        )

    def forward(self, x):

        return self.block(x)


# ============================================================
# 9. SIAMESE U-NET
# ============================================================

class SiameseUNet(nn.Module):

    def __init__(self):

        super().__init__()

        # ----------------------------------------------------
        # Shared encoder
        # ----------------------------------------------------

        self.enc1 = DoubleConv(
            3, 32
        )

        self.enc2 = DoubleConv(
            32, 64
        )

        self.enc3 = DoubleConv(
            64, 128
        )

        self.enc4 = DoubleConv(
            128, 256
        )

        self.pool = nn.MaxPool2d(
            2
        )

        # ----------------------------------------------------
        # Decoder
        #
        # 32x32
        #   ↓
        # 64x64
        #   ↓
        # 128x128
        #   ↓
        # 256x256
        #
        # IMPORTANT:
        # There is NO additional upsampling after 256x256.
        # ----------------------------------------------------

        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.dec3 = DoubleConv(
            256,
            128
        )

        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.dec2 = DoubleConv(
            128,
            64
        )

        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            kernel_size=2,
            stride=2
        )

        self.dec1 = DoubleConv(
            64,
            32
        )

        # ----------------------------------------------------
        # Final segmentation layer
        # ----------------------------------------------------

        self.mask_head = nn.Conv2d(
            32,
            1,
            kernel_size=1
        )

        # ----------------------------------------------------
        # Classification head
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.AdaptiveAvgPool2d(
                1
            ),

            nn.Flatten(),

            nn.Linear(
                256,
                64
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(
                0.3
            ),

            nn.Linear(
                64,
                1
            )
        )

    def encode(self, x):

        e1 = self.enc1(x)

        p1 = self.pool(e1)

        e2 = self.enc2(p1)

        p2 = self.pool(e2)

        e3 = self.enc3(p2)

        p3 = self.pool(e3)

        e4 = self.enc4(p3)

        return e1, e2, e3, e4

    def forward_single(
        self,
        x
    ):

        e1, e2, e3, e4 = self.encode(
            x
        )

        return e1, e2, e3, e4

    def forward(
        self,
        image_a,
        image_b
    ):

        # ----------------------------------------------------
        # Siamese encoder
        # Same weights for A and B
        # ----------------------------------------------------

        a1, a2, a3, a4 = self.encode(
            image_a
        )

        b1, b2, b3, b4 = self.encode(
            image_b
        )

        # ----------------------------------------------------
        # Difference features
        # ----------------------------------------------------

        d1 = torch.abs(
            a1 - b1
        )

        d2 = torch.abs(
            a2 - b2
        )

        d3 = torch.abs(
            a3 - b3
        )

        d4 = torch.abs(
            a4 - b4
        )

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        classification = self.classifier(
            d4
        ).squeeze(1)

        # ----------------------------------------------------
        # Decoder
        # ----------------------------------------------------

        x = self.up3(
            d4
        )

        x = torch.cat(
            [x, d3],
            dim=1
        )

        x = self.dec3(
            x
        )

        x = self.up2(
            x
        )

        x = torch.cat(
            [x, d2],
            dim=1
        )

        x = self.dec2(
            x
        )

        x = self.up1(
            x
        )

        x = torch.cat(
            [x, d1],
            dim=1
        )

        x = self.dec1(
            x
        )

        # ----------------------------------------------------
        # IMPORTANT:
        # x is now exactly [B,32,256,256]
        # ----------------------------------------------------

        mask = self.mask_head(
            x
        )

        return (
            mask,
            classification,
            d4
        )


# ============================================================
# 10. CAPTION DECODER
# ============================================================

class CaptionDecoder(nn.Module):

    def __init__(
        self,
        feature_dim,
        vocab_size,
        hidden_dim=256
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            hidden_dim,
            padding_idx=PAD_IDX
        )

        self.feature_projection = nn.Linear(
            feature_dim,
            hidden_dim
        )

        self.gru = nn.GRU(
            hidden_dim,
            hidden_dim,
            batch_first=True
        )

        self.output = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(
        self,
        features,
        captions
    ):

        # features:
        # [B,256,32,32]

        features = F.adaptive_avg_pool2d(
            features,
            1
        )

        features = features.flatten(
            1
        )

        hidden = self.feature_projection(
            features
        ).unsqueeze(0)

        # Remove final token for teacher forcing
        inputs = captions[:, :-1]

        embeddings = self.embedding(
            inputs
        )

        outputs, _ = self.gru(
            embeddings,
            hidden
        )

        logits = self.output(
            outputs
        )

        return logits


# ============================================================
# 11. COMPLETE MODEL
# ============================================================

class ChangeCaptionModel(nn.Module):

    def __init__(
        self,
        vocab_size
    ):

        super().__init__()

        self.siamese = SiameseUNet()

        self.caption_decoder = CaptionDecoder(
            feature_dim=256,
            vocab_size=vocab_size
        )

    def forward(
        self,
        image_a,
        image_b,
        captions
    ):

        mask, classification, features = \
            self.siamese(
                image_a,
                image_b
            )

        caption_logits = \
            self.caption_decoder(
                features,
                captions
            )

        return (
            mask,
            classification,
            caption_logits
        )


# ============================================================
# 12. MODEL
# ============================================================

model = ChangeCaptionModel(
    VOCAB_SIZE
).to(DEVICE)

params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "\nTrainable parameters:",
    round(params / 1e6, 3),
    "M"
)

# ============================================================
# 13. VERIFY BATCH + MODEL SHAPES
# ============================================================

print("\n" + "=" * 70)
print("BATCH VERIFICATION")
print("=" * 70)

batch = next(
    iter(train_loader)
)

image_a = batch["image_a"].to(
    DEVICE
)

image_b = batch["image_b"].to(
    DEVICE
)

mask = batch["mask"].to(
    DEVICE
)

changeflag = batch["changeflag"].to(
    DEVICE
)

captions = batch["caption"].to(
    DEVICE
)

print("Image A:", image_a.shape)
print("Image B:", image_b.shape)
print("Mask:", mask.shape)
print("Change flag:", changeflag.shape)
print("Caption:", captions.shape)

model.eval()

with torch.no_grad():

    pred_mask, pred_flag, pred_caption = model(
        image_a[:2],
        image_b[:2],
        captions[:2]
    )

print(
    "Predicted mask:",
    pred_mask.shape
)

print(
    "Predicted classification:",
    pred_flag.shape
)

print(
    "Predicted captions:",
    pred_caption.shape
)

assert pred_mask.shape == torch.Size(
    [2, 1, 256, 256]
), f"Wrong mask shape: {pred_mask.shape}"

assert pred_flag.shape == torch.Size(
    [2]
), f"Wrong classification shape: {pred_flag.shape}"

print("\nMODEL SHAPE VERIFICATION: PASS")

del batch
del image_a
del image_b
del mask
del changeflag
del captions
del pred_mask
del pred_flag
del pred_caption

torch.cuda.empty_cache()

# ============================================================
# 14. LOSSES
# ============================================================

bce_seg = nn.BCEWithLogitsLoss()

bce_cls = nn.BCEWithLogitsLoss()

caption_loss_fn = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)


def dice_loss(
    logits,
    targets,
    smooth=1.0
):

    probs = torch.sigmoid(
        logits
    )

    probs = probs.contiguous().view(
        probs.size(0),
        -1
    )

    targets = targets.contiguous().view(
        targets.size(0),
        -1
    )

    intersection = (
        probs * targets
    ).sum(dim=1)

    dice = (
        2 * intersection + smooth
    ) / (
        probs.sum(dim=1)
        + targets.sum(dim=1)
        + smooth
    )

    return (
        1 - dice
    ).mean()


def segmentation_loss(
    pred,
    target
):

    bce = bce_seg(
        pred,
        target
    )

    dice = dice_loss(
        pred,
        target
    )

    return bce + dice


# ============================================================
# 15. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5,
    min_lr=1e-7
)

# ============================================================
# 16. AMP
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

# ============================================================
# 17. TRAIN ONE EPOCH
# ============================================================

def train_one_epoch():

    model.train()

    total_loss = 0
    total_seg = 0
    total_cls = 0
    total_cap = 0

    pbar = tqdm(
        train_loader,
        desc="Training"
    )

    for batch in pbar:

        image_a = batch["image_a"].to(
            DEVICE,
            non_blocking=True
        )

        image_b = batch["image_b"].to(
            DEVICE,
            non_blocking=True
        )

        mask = batch["mask"].to(
            DEVICE,
            non_blocking=True
        )

        changeflag = batch["changeflag"].to(
            DEVICE,
            non_blocking=True
        )

        captions = batch["caption"].to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(DEVICE.type == "cuda")
        ):

            pred_mask, pred_flag, pred_caption = model(
                image_a,
                image_b,
                captions
            )

            # -----------------------------
            # Segmentation
            # -----------------------------

            loss_seg = segmentation_loss(
                pred_mask,
                mask
            )

            # -----------------------------
            # Classification
            # -----------------------------

            loss_cls = bce_cls(
                pred_flag,
                changeflag
            )

            # -----------------------------
            # Caption
            # -----------------------------

            target_caption = captions[:, 1:]

            loss_cap = caption_loss_fn(
                pred_caption.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                target_caption.reshape(
                    -1
                )
            )

            # -----------------------------
            # Total
            # -----------------------------

            loss = (
                SEG_WEIGHT * loss_seg
                +
                CLS_WEIGHT * loss_cls
                +
                CAP_WEIGHT * loss_cap
            )

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        total_loss += loss.item()
        total_seg += loss_seg.item()
        total_cls += loss_cls.item()
        total_cap += loss_cap.item()

        pbar.set_postfix(
            loss=f"{loss.item():.3f}"
        )

    n = len(train_loader)

    return {
        "loss": total_loss / n,
        "seg": total_seg / n,
        "cls": total_cls / n,
        "cap": total_cap / n
    }


# ============================================================
# 18. VALIDATION
# ============================================================

@torch.no_grad()
def validate():

    model.eval()

    total_loss = 0
    total_seg = 0
    total_cls = 0
    total_cap = 0

    correct = 0
    total = 0

    pbar = tqdm(
        val_loader,
        desc="Validation"
    )

    for batch in pbar:

        image_a = batch["image_a"].to(
            DEVICE,
            non_blocking=True
        )

        image_b = batch["image_b"].to(
            DEVICE,
            non_blocking=True
        )

        mask = batch["mask"].to(
            DEVICE,
            non_blocking=True
        )

        changeflag = batch["changeflag"].to(
            DEVICE,
            non_blocking=True
        )

        captions = batch["caption"].to(
            DEVICE,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(DEVICE.type == "cuda")
        ):

            pred_mask, pred_flag, pred_caption = model(
                image_a,
                image_b,
                captions
            )

            loss_seg = segmentation_loss(
                pred_mask,
                mask
            )

            loss_cls = bce_cls(
                pred_flag,
                changeflag
            )

            target_caption = captions[:, 1:]

            loss_cap = caption_loss_fn(
                pred_caption.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                target_caption.reshape(
                    -1
                )
            )

            loss = (
                SEG_WEIGHT * loss_seg
                +
                CLS_WEIGHT * loss_cls
                +
                CAP_WEIGHT * loss_cap
            )

        total_loss += loss.item()
        total_seg += loss_seg.item()
        total_cls += loss_cls.item()
        total_cap += loss_cap.item()

        predictions = (
            torch.sigmoid(pred_flag)
            >= 0.5
        ).float()

        correct += (
            predictions == changeflag
        ).sum().item()

        total += changeflag.numel()

    n = len(val_loader)

    return {
        "loss": total_loss / n,
        "seg": total_seg / n,
        "cls": total_cls / n,
        "cap": total_cap / n,
        "acc": correct / total
    }


# ============================================================
# 19. TRAINING
# ============================================================

print("\n" + "=" * 70)
print("STARTING EXPERIMENT 2")
print("=" * 70)

print(
    "Changes:"
)

print(
    "Segmentation loss = BCE + Dice"
)

print(
    "Segmentation weight =",
    SEG_WEIGHT
)

print(
    "Classification weight =",
    CLS_WEIGHT
)

print(
    "Caption weight =",
    CAP_WEIGHT
)

print(
    "Epochs =",
    EPOCHS
)

history = []

best_val_loss = float("inf")
best_epoch = 0

for epoch in range(
    1,
    EPOCHS + 1
):

    print(
        f"\nEpoch {epoch}/{EPOCHS}"
    )

    train_metrics = train_one_epoch()

    val_metrics = validate()

    scheduler.step(
        val_metrics["loss"]
    )

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"\nTrain: {train_metrics['loss']:.4f}"
        f" | Seg {train_metrics['seg']:.4f}"
        f" | Cls {train_metrics['cls']:.4f}"
        f" | Cap {train_metrics['cap']:.4f}"
    )

    print(
        f"Val: {val_metrics['loss']:.4f}"
        f" | Seg {val_metrics['seg']:.4f}"
        f" | Cls {val_metrics['cls']:.4f}"
        f" | Cap {val_metrics['cap']:.4f}"
        f" | Acc {val_metrics['acc']:.4f}"
    )

    print(
        f"LR: {current_lr:.2e}"
    )

    history.append({

        "epoch": epoch,

        "train_loss":
            train_metrics["loss"],

        "train_seg":
            train_metrics["seg"],

        "train_cls":
            train_metrics["cls"],

        "train_cap":
            train_metrics["cap"],

        "val_loss":
            val_metrics["loss"],

        "val_seg":
            val_metrics["seg"],

        "val_cls":
            val_metrics["cls"],

        "val_cap":
            val_metrics["cap"],

        "val_acc":
            val_metrics["acc"],

        "lr":
            current_lr
    })

    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    if val_metrics["loss"] < best_val_loss:

        best_val_loss = val_metrics["loss"]

        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "scheduler_state_dict":
                    scheduler.state_dict(),

                "val_loss":
                    best_val_loss,

                "vocab_size":
                    VOCAB_SIZE,

                "stoi":
                    stoi,

                "itos":
                    itos
            },
            BEST_MODEL_PATH
        )

        print(
            f"✓ Best model saved "
            f"(epoch {epoch})"
        )

    # --------------------------------------------------------
    # Save history continuously
    # --------------------------------------------------------

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False
    )


# ============================================================
# 20. COMPLETE
# ============================================================

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    "Best validation loss:",
    best_val_loss
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Saved:"
)

print(
    " ",
    BEST_MODEL_PATH
)

print(
    " ",
    HISTORY_PATH
)

DEVICE
Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU

Dataset:
Total: 7264
split
train    6345
val       919
Name: count, dtype: int64

Change distribution:
split  changeflag
train  0             2938
       1             3407
val    0              452
       1              467
dtype: int64

Vocabulary size: 860

Train samples: 6345
Val samples: 919
Train batches: 1587
Val batches: 230

Trainable parameters: 2.845 M

BATCH VERIFICATION
Image A: torch.Size([4, 3, 256, 256])
Image B: torch.Size([4, 3, 256, 256])
Mask: torch.Size([4, 1, 256, 256])
Change flag: torch.Size([4])
Caption: torch.Size([4, 40])
Predicted mask: torch.Size([2, 1, 256, 256])
Predicted classification: torch.Size([2])
Predicted captions: torch.Size([2, 39, 860])

MODEL SHAPE VERIFICATION: PASS

STARTING EXPERIMENT 2
Changes:
Segmentation loss = BCE + Dice
Segmentation weight = 2.0
Classification weight = 1.0
Caption weight = 1.0
Epochs = 70

Epoch 1/70


Training:   0%|          | 0/1587 [00:00<?, ?it/s]

Validation:   0%|          | 0/230 [00:00<?, ?it/s]


Train: 5.1057 | Seg 1.2122 | Cls 0.5721 | Cap 2.1092
Val: 4.2236 | Seg 1.1429 | Cls 0.6170 | Cap 1.3208 | Acc 0.6942
LR: 1.00e-04
✓ Best model saved (epoch 1)

Epoch 2/70


Training:   0%|          | 0/1587 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/230 [00:00<?, ?it/s]


Train: 3.0382 | Seg 1.1354 | Cls 0.1289 | Cap 0.6385
Val: 3.7893 | Seg 1.1349 | Cls 0.5507 | Cap 0.9688 | Acc 0.8041
LR: 1.95e-07

Epoch 70/70


Training:   0%|          | 0/1587 [00:00<?, ?it/s]

Validation:   0%|          | 0/230 [00:00<?, ?it/s]


Train: 3.0256 | Seg 1.1351 | Cls 0.1151 | Cap 0.6403
Val: 3.6436 | Seg 1.1340 | Cls 0.4467 | Cap 0.9288 | Acc 0.8400
LR: 1.95e-07

TRAINING COMPLETE
Best validation loss: 3.5814030792402183
Best epoch: 24
Saved:
  experiment2_best_model.pth
  experiment2_training_history.csv


In [5]:
import torch

CHECKPOINT = "experiment2_best_model.pth"

ckpt = torch.load(
    CHECKPOINT,
    map_location="cpu"
)

state = ckpt["model_state_dict"]

print("=" * 70)
print("CHECKPOINT ARCHITECTURE")
print("=" * 70)

for k, v in state.items():
    print(f"{k:55s} {tuple(v.shape)}")

CHECKPOINT ARCHITECTURE
siamese.enc1.block.0.weight                             (32, 3, 3, 3)
siamese.enc1.block.1.weight                             (32,)
siamese.enc1.block.1.bias                               (32,)
siamese.enc1.block.1.running_mean                       (32,)
siamese.enc1.block.1.running_var                        (32,)
siamese.enc1.block.1.num_batches_tracked                ()
siamese.enc1.block.3.weight                             (32, 32, 3, 3)
siamese.enc1.block.4.weight                             (32,)
siamese.enc1.block.4.bias                               (32,)
siamese.enc1.block.4.running_mean                       (32,)
siamese.enc1.block.4.running_var                        (32,)
siamese.enc1.block.4.num_batches_tracked                ()
siamese.enc2.block.0.weight                             (64, 32, 3, 3)
siamese.enc2.block.1.weight                             (64,)
siamese.enc2.block.1.bias                               (64,)
siamese.enc2.block.1.runni